**Molecular physical pharmacy, 3FC003**

**Molecular dynamics exercise**

# High-throughput peptide self-assembly

**Introduction**

This exercise will introduce you to combinatorial screening for peptide self-assembly using the Martini force field for proteins. The process is automated using a number of bash scripts, Gromacs tools and scripting capabilities within the visual molecular dynamics (VMD) program. After the simulations, visual inspection is done using VMD and analysis of the assembled structures is done using Gromacs tools.

NOTE that the exercise is written for versions 5.1 or 2016/2018 of Gromacs and will show errors when used with earlier versions

# Important:

Before start with this lab click in the edit menu and select clear all output

In [2]:
# Cloning the course repository
!git clone https://github.com/computationalpharmaceutics/3FC003.git

Cloning into '3FC003'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 9 (delta 0), reused 9 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 2.60 MiB | 12.67 MiB/s, done.


In [3]:
# Installing GROMACS with CPU support only
!apt install gromacs &> /dev/null

In [4]:
# Confirm it's installed
!gmx -version

              :-) GROMACS - gmx, 2023.3-Ubuntu_2023.3_1ubuntu3 (-:

Executable:   /usr/bin/gmx
Data prefix:  /usr
Working dir:  /content
Command line:
  gmx -version

GROMACS version:    2023.3-Ubuntu_2023.3_1ubuntu3
Precision:          mixed
Memory model:       64 bit
MPI library:        thread_mpi
OpenMP support:     enabled (GMX_OPENMP_MAX_THREADS = 128)
GPU support:        disabled
SIMD instructions:  SSE4.1
CPU FFT library:    fftw-3.3.10-sse2-avx
GPU FFT library:    none
Multi-GPU FFT:      none
RDTSCP usage:       enabled
TNG support:        enabled
Hwloc support:      hwloc-2.8.0
Tracing support:    disabled
C compiler:         /usr/bin/cc GNU 13.2.0
C compiler flags:   -fexcess-precision=fast -funroll-all-loops -msse4.1 -Wno-missing-field-initializers -O3 -DNDEBUG
C++ compiler:       /usr/bin/c++ GNU 13.2.0
C++ compiler flags: -fexcess-precision=fast -funroll-all-loops -msse4.1 -Wno-missing-field-initializers -Wno-cast-function-type-strict SHELL:-fopenmp -O3 -DNDEBUG
BLAS libr

In [ ]:
import os
os.chdir('/content/3FG003/HT_peptide_self_assembly')
!ls -l

In [ ]:
# Seting the path to GROMACS
import os
os.environ["PATH"] += ":/usr/local/gromacs/bin"

# Looking for the working directory
!pwd

In [ ]:
# Changing the working directory to

import os
os.chdir('/content/3FG003/HT_peptide_self_assembly')
!pwd

In [ ]:
# Installing necessary packages
!pip install py3Dmol

# 1 Introduction

The exercise uses the Martini coarse-grained protein force field to screen short peptides for self-assembly. Bash scripts, GROMACS tools, and VMD scripting automate the system-building and simulation stages. VMD is used for structural inspection in the original handout; this notebook adds browser-based `py3Dmol` views so the same checkpoints are visible in Colab.

The five numbered directories mirror the scientific workflow:

| Directory | Purpose | Successful checkpoint |
|---|---|---|
| `1_Background` | Papers and optional further reading | - |
| `2_Creating_coordinates` | Build 20 atomistic Tyr-X-Tyr structures | `2_Done` |
| `3_Coarse-graining` | Martinize, pack, solvate, neutralize | `3_Done` |
| `4_Running_simulations` | Energy minimization and 12.5 ns MD | `4_Done` |
| `5_Analysis` | Visual inspection, SASA, clustering, and MOI | `5_Done` |

The purpose is to practise the commands for **one** peptide before applying the automation scripts to all 20.

In [ ]:
# Colab-only software setup. The course archive itself supplies all force-field files.
import importlib.util
import shutil
import subprocess
import sys

google_spec = importlib.util.find_spec("google")
IN_COLAB = google_spec is not None and importlib.util.find_spec("google.colab") is not None

if importlib.util.find_spec("py3Dmol") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "py3Dmol"], check=True)

# Current GROMACS is useful for many file-processing steps. The supplied input was
# authored for GROMACS 5.1/2016/2018, so always read compatibility messages.
if IN_COLAB and shutil.which("gmx") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "gromacs"], check=True)

print("Running in Colab:", IN_COLAB)
if shutil.which("gmx"):
    version = subprocess.run(["gmx", "--version"], text=True, capture_output=True).stdout.splitlines()
    print(version[0] if version else "GROMACS found")
else:
    print("GROMACS not found; use the supplied *_Done checkpoints.")

## 1.1 Unpack the course material

The handout starts with:

```bash
tar -xvf Peptide_assembly.tgz
```

Upload the archive below. The extraction cell performs the same operation with a path-safety check and locates `Peptide_assembly_GMX5-2016` automatically.

In [ ]:
from pathlib import Path
import tarfile

CONTENT = Path("/content") if IN_COLAB else Path.cwd()
ARCHIVE = CONTENT / "Peptide_assembly.tgz"

if not ARCHIVE.exists():
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.endswith((".tgz", ".tar.gz"))]
        if not candidates:
            raise FileNotFoundError("Upload Peptide_assembly.tgz to continue.")
        source = candidates[0]
        if source.resolve() != ARCHIVE.resolve():
            shutil.move(str(source), ARCHIVE)
    else:
        raise FileNotFoundError(f"Place Peptide_assembly.tgz at {ARCHIVE}")

def safe_extract_tgz(archive, destination):
    destination = destination.resolve()
    # The supplied file uses a .tgz suffix but some course copies are plain tar.
    with tarfile.open(archive, "r:*") as tf:
        for member in tf.getmembers():
            target = (destination / member.name).resolve()
            if destination not in target.parents and target != destination:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        tf.extractall(destination)

safe_extract_tgz(ARCHIVE, CONTENT)
print("Extracted:", ARCHIVE)

In [ ]:
roots = [p for p in CONTENT.rglob("Peptide_assembly_GMX5-2016") if p.is_dir()]
if len(roots) != 1:
    raise RuntimeError(f"Expected one extracted course directory, found: {roots}")
ROOT = roots[0]

required = [
    "1_Background", "2_Creating_coordinates", "2_Done",
    "3_Coarse-graining", "3_Done", "4_Running_simulations",
    "4_Done", "5_Analysis", "5_Done",
]
missing = [name for name in required if not (ROOT / name).is_dir()]
if missing:
    raise FileNotFoundError(f"Missing course directories: {missing}")

for name in required:
    print(f"{name:25s} {sum(1 for p in (ROOT / name).iterdir() if p.is_file()):4d} files")

## 1.2 Optional background papers

The `1_Background` directory contains the papers supplied with the lab. Reading them is optional for completing the commands, but the two Ray papers are central to the experimental motivation.

In [ ]:
for path in sorted((ROOT / "1_Background").glob("*.pdf")):
    print(path.name)

# Background

Short oligopeptides can self-assemble into nanostructures relevant to biomedicine, food science, cosmetics, and nanotechnology. Twenty amino acids create a very large design space, so computation can prioritize candidates before synthesis.

Two design problems motivate the screen:

- hydrophobic interactions promote assembly in water but can also reduce solubility;
- simple sequence rules do not reliably distinguish ordered nanostructures from disordered precipitation.

Peptide nanomaterials are often terminally protected. N-terminal groups include acetyl, Fmoc, naphthalene, pyrene, and t-Boc; common C-terminal modifications include amides and esters. These modifications add interactions and remove terminal charge repulsions.

Ray and co-workers reported hollow nanotubes with an inner diameter of about 5 Å for protected `Boc-Tyr-X-Tyr-OMe` tripeptides where X was Val or Ile. Mutating either terminal Tyr prevented nanotube formation. This raises two questions: what happens for the other 18 middle residues, and do these structures remain stable in water rather than a water/methanol crystallization environment?

In [ ]:
# Visual summary of the chemical simplification made by the exercise.
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.set_xlim(0, 11)
ax.set_ylim(0, 3)
ax.axis("off")

def box(x, label, color):
    patch = FancyBboxPatch((x, 1.05), 1.55, 0.8, boxstyle="round,pad=0.12",
                           facecolor=color, edgecolor="#203040", linewidth=1.4)
    ax.add_patch(patch)
    ax.text(x + 0.775, 1.45, label, ha="center", va="center", fontsize=12, weight="bold")

box(0.25, "Boc", "#f6c85f")
box(2.0, "Tyr", "#6f4e9c")
box(3.75, "X", "#9fd3c7")
box(5.5, "Tyr", "#6f4e9c")
box(7.25, "OMe", "#f6c85f")
ax.add_patch(FancyArrowPatch((8.95, 1.45), (9.75, 1.45), arrowstyle="->", mutation_scale=18))
ax.text(10.35, 1.45, "Tyr-X-Tyr\nneutral termini", ha="center", va="center", fontsize=11)
ax.text(4.65, 2.45, "Experimental protected tripeptide", ha="center", fontsize=12, weight="bold")
ax.text(10.05, 2.45, "Martini model used here", ha="center", fontsize=12, weight="bold")
plt.show()

> **Question 1 - model design.** What interactions are lost when `Boc-Tyr-X-Tyr-OMe` is simplified to Tyr-X-Tyr with neutral termini? Why might this matter when comparing an aqueous simulation to crystallization from water/methanol?

# 2 Creating peptide coordinates

The model is Tyr-X-Tyr with uncharged termini, and X is varied across all 20 standard amino acids. The course workflow uses a backbone-only template (`XYZ.pdb`), edits it for each sequence, and calls VMD with `create_tripeptides.tcl` to graft the side chains.

Choose one peptide to follow manually. Val is the default because it was one of the experimentally reported nanotube-forming middle residues.

In [ ]:
AMINO_ACIDS = [
    "ALA", "CYS", "ASP", "GLU", "PHE", "GLY", "HIS", "ILE", "LYS", "LEU",
    "MET", "ASN", "PRO", "GLN", "ARG", "SER", "THR", "VAL", "TRP", "TYR",
]
MIDDLE_RESIDUE = "VAL"  # @param ["ALA", "CYS", "ASP", "GLU", "PHE", "GLY", "HIS", "ILE", "LYS", "LEU", "MET", "ASN", "PRO", "GLN", "ARG", "SER", "THR", "VAL", "TRP", "TYR"]
if MIDDLE_RESIDUE not in AMINO_ACIDS:
    raise ValueError("Choose one of the 20 three-letter residue codes.")

PEPTIDE = f"TYR-{MIDDLE_RESIDUE}-TYR"
COORD_DIR = ROOT / "2_Creating_coordinates"
CG_DIR = ROOT / "3_Coarse-graining"
SIM_DIR = ROOT / "4_Running_simulations"
ANALYSIS_DIR = ROOT / "5_Analysis"
print("Selected peptide:", PEPTIDE)

## 2.1 Inspect and configure `make_peptides.sh`

Enter `2_Creating_coordinates` and inspect the script. The VMD executable assignment must read:

```bash
vmd='vmd'
```

If VMD has another path on your computer, edit that line before continuing.

In [ ]:
make_script = COORD_DIR / "make_peptides.sh"
print(make_script.read_text())

assignments = [line.strip() for line in make_script.read_text().splitlines() if line.strip().startswith("vmd=")]
print("\nDetected VMD assignment:", assignments or "not found")

## 2.2 Make the script executable and generate all 20 structures

The handout command is:

```bash
./make_peptides.sh
```

VMD is normally unavailable in a Colab CPU runtime. Leave the switch `False` in Colab and the next cell will restore the successful atomistic checkpoint. On a local system with VMD installed, set it to `True`.

In [ ]:
RUN_VMD_COORDINATE_GENERATION = False  # @param {type:"boolean"}
make_script.chmod(make_script.stat().st_mode | 0o111)

if RUN_VMD_COORDINATE_GENERATION:
    if shutil.which("vmd") is None:
        raise RuntimeError("VMD is not on PATH. Edit vmd=... in make_peptides.sh or use 2_Done.")
    subprocess.run(["./make_peptides.sh"], cwd=COORD_DIR, check=True)
else:
    print("VMD generation skipped. The exact command is: ./make_peptides.sh")

In [ ]:
# Compatibility checkpoint: copy only missing successful structures into the working directory.
generated = sorted(COORD_DIR.glob("TYR-???-TYR_aa.pdb"))
if len(generated) != 20:
    for source in (ROOT / "2_Done").glob("TYR-???-TYR_aa.pdb"):
        target = COORD_DIR / source.name
        if not target.exists():
            shutil.copy2(source, target)
    generated = sorted(COORD_DIR.glob("TYR-???-TYR_aa.pdb"))

if len(generated) != 20:
    raise RuntimeError(f"Expected 20 atomistic structures; found {len(generated)}")
print("Generated/restored structures:", len(generated))
print("\n".join(path.name for path in generated))

## 2.3 Inspect an atomistic structure

The handout writes `vmd TYR-XXX-TYR.pdb`; the supplied generator actually names its atomistic outputs `TYR-XXX-TYR_aa.pdb`, matching the martinize command in the next section. Locally, use:

```bash
vmd TYR-XXX-TYR_aa.pdb
```

The interactive view below is the Colab equivalent.

In [ ]:
import py3Dmol

def view_pdb(path, width=760, height=440):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(Path(path).read_text(), "pdb")
    view.setStyle({"stick": {"radius": 0.18}, "sphere": {"scale": 0.28}})
    view.setBackgroundColor("white")
    view.zoomTo()
    return view.show()

atomistic_path = COORD_DIR / f"{PEPTIDE}_aa.pdb"
view_pdb(atomistic_path)

> **Question 2 - atomistic structure.** Identify the backbone, the two aromatic Tyr side chains, and the side chain of X. Is X hydrophobic, polar, acidic, or basic? Predict how it could alter association even though terminal Tyr residues are present in every sequence.

## 2.4 How the coordinate builder works

This detail is optional in the handout but important for understanding the automation:

1. `XYZ.pdb` supplies a generic atomistic tripeptide backbone.
2. `make_peptides.sh` loops through the 20 middle residues.
3. The shell script edits the template and `create_tripeptides.tcl` for the current sequence.
4. VMD/psfgen grafts the requested side chains and writes the PDB file.

Because the residue lists are arrays in the Bash script, the same pattern can be extended to a much larger combinatorial library.

In [ ]:
print("Backbone template (first 15 lines):")
print("\n".join((COORD_DIR / "XYZ.pdb").read_text().splitlines()[:15]))
print("\nVMD instruction script:")
print((COORD_DIR / "create_tripeptides.tcl").read_text())

# 3 Create a coarse-grained system with `martinize.py` and GROMACS

This section produces:

1. a Martini topology for one peptide;
2. a coarse-grained single-peptide structure;
3. an 8 x 8 x 8 nm box containing 100 peptides;
4. Martini water and, when required, counterions.

## 3.1 Martinize one neutral, extended tripeptide

Replace `XXX` with your selected residue and run this **as one line** from `3_Coarse-graining`:

```bash
./martinize.py -f ../2_Creating_coordinates/TYR-XXX-TYR_aa.pdb -name TYR-XXX-TYR -o TYR-XXX-TYR.top -x TYR-XXX-TYR.pdb -ff martini22 -nt -ss EEE
```

`-nt` requests neutral termini. `-ss EEE` assigns the extended beta-region secondary structure reported for the crystal backbone. The bundled `martinize.py` is a legacy Python 2 program, so its successful outputs are used by default in modern Colab.

In [ ]:
RUN_LEGACY_MARTINIZE = False  # @param {type:"boolean"}
martinize_command = [
    "./martinize.py", "-f", f"../2_Creating_coordinates/{PEPTIDE}_aa.pdb",
    "-name", PEPTIDE, "-o", f"{PEPTIDE}.top", "-x", f"{PEPTIDE}.pdb",
    "-ff", "martini22", "-nt", "-ss", "EEE",
]
print("Resolved command:\n", " ".join(martinize_command))

if RUN_LEGACY_MARTINIZE:
    if shutil.which("python2") is None:
        raise RuntimeError("The supplied martinize.py requires Python 2. Use 3_Done or a compatible local environment.")
    subprocess.run(["python2", *martinize_command], cwd=CG_DIR, check=True)
else:
    for suffix in (".top", ".itp", ".pdb"):
        shutil.copy2(ROOT / "3_Done" / f"{PEPTIDE}{suffix}", CG_DIR / f"{PEPTIDE}{suffix}")
    print("Restored the successful martinize outputs from 3_Done.")

## 3.2 Correct the Martini force-field include

Open `TYR-XXX-TYR.top` and ensure its main include is exactly:

```c
#include "martini_v2.2.itp"
```

The next cell performs that explicit text edit and prints the result.

In [ ]:
top_path = CG_DIR / f"{PEPTIDE}.top"
top_lines = top_path.read_text().splitlines()
replaced = False
for i, line in enumerate(top_lines):
    if line.lstrip().startswith("#include") and "martini_v2." in line and "ions" not in line:
        top_lines[i] = '#include "martini_v2.2.itp"'
        replaced = True
        break
if not replaced:
    top_lines.insert(0, '#include "martini_v2.2.itp"')
top_path.write_text("\n".join(top_lines).rstrip() + "\n")
print(top_path.read_text())

## 3.3 Inspect the bead mapping

The generated `.itp` defines the molecule, its bead types, and bonded interactions. Inspect `[ atoms ]` and ask whether the bead identities preserve the backbone and side-chain chemistry that should matter for aggregation.

In [ ]:
def read_section(path, section_name):
    result = []
    active = False
    for line in Path(path).read_text().splitlines():
        stripped = line.strip()
        if stripped.startswith("[") and stripped.endswith("]"):
            active = stripped.strip("[] ").lower() == section_name.lower()
            continue
        if active:
            if stripped.startswith("["):
                break
            result.append(line)
    return "\n".join(result).strip()

itp_path = CG_DIR / f"{PEPTIDE}.itp"
print("[ atoms ]")
print(read_section(itp_path, "atoms"))

In [ ]:
def gro_to_pdb(path, include_solvent=False):
    lines = Path(path).read_text().splitlines()
    atom_count = int(lines[1].strip())
    pdb_lines = []
    serial = 1
    for line in lines[2:2 + atom_count]:
        resid = int(line[0:5])
        resname = line[5:10].strip()
        atomname = line[10:15].strip()
        if not include_solvent and resname in {"W", "NA+", "CL-"}:
            continue
        x, y, z = (10.0 * float(line[a:b]) for a, b in ((20, 28), (28, 36), (36, 44)))
        pdb_lines.append(
            f"HETATM{serial:5d} {atomname[:4]:>4s} {resname[:3]:>3s} A{resid % 10000:4d}    "
            f"{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           C"
        )
        serial += 1
    return "\n".join(pdb_lines) + "\nEND\n"

def view_gro(path, width=760, height=480, sphere_scale=0.55):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(gro_to_pdb(path), "pdb")
    view.setStyle({"sphere": {"scale": sphere_scale, "colorscheme": "Jmol"}})
    view.setBackgroundColor("white")
    view.zoomTo()
    return view.show()

cg_pdb = CG_DIR / f"{PEPTIDE}.pdb"
view_pdb(cg_pdb)

> **Question 3 - Martini mapping.** Which bead types represent the backbone and the three side chains? Do the choices make chemical sense for your selected X? What atomistic detail is necessarily lost in a coarse-grained representation?

## 3.4 Insert 100 peptides in an 8 nm box

From `3_Coarse-graining`, run:

```bash
gmx insert-molecules -box 8 8 8 -nmol 100 -ci TYR-XXX-TYR.pdb -radius 0.4 -o TYR-XXX-TYR_box.gro
```

The `0.4` nm insertion radius keeps the peptides somewhat separated at the start.

## 3.5 Solvate with the equilibrated Martini water box

```bash
gmx solvate -cp TYR-XXX-TYR_box.gro -cs water-80A_eq.gro -radius 0.21 -o TYR-XXX-TYR_water.gro
```

The `0.21` nm solvent radius gives an approximately suitable Martini-water density. The supplied `water-80A_eq.gro` already matches the 8 nm peptide box. For a substantially different box, first equilibrate a water box of the desired size; otherwise `solvate` can insert too many water beads.

In [ ]:
RUN_GROMACS_SYSTEM_BUILD = False  # @param {type:"boolean"}

insert_command = [
    "gmx", "insert-molecules", "-box", "8", "8", "8", "-nmol", "100",
    "-ci", f"{PEPTIDE}.pdb", "-radius", "0.4", "-o", f"{PEPTIDE}_box.gro",
]
solvate_command = [
    "gmx", "solvate", "-cp", f"{PEPTIDE}_box.gro", "-cs", "water-80A_eq.gro",
    "-radius", "0.21", "-o", f"{PEPTIDE}_water.gro",
]
print("Resolved insertion command:\n", " ".join(insert_command))
print("Resolved solvation command:\n", " ".join(solvate_command))

if RUN_GROMACS_SYSTEM_BUILD:
    if shutil.which("gmx") is None:
        raise RuntimeError("GROMACS is not available.")
    subprocess.run(insert_command, cwd=CG_DIR, check=True)
    subprocess.run(solvate_command, cwd=CG_DIR, check=True)
else:
    for suffix in ("_box.gro", "_water.gro", "_genion.tpr"):
        shutil.copy2(ROOT / "3_Done" / f"{PEPTIDE}{suffix}", CG_DIR / f"{PEPTIDE}{suffix}")
    print("Restored the packed/solvated checkpoint from 3_Done.")

In [ ]:
# Visualization checkpoint: 100 initially dispersed peptides; water and ions are hidden.
view_gro(CG_DIR / f"{PEPTIDE}_water.gro", sphere_scale=0.42)

## 3.6 Edit `[ molecules ]`: 100 peptides and the actual water count

The topology must match the coordinate file exactly. Set the peptide count to 100, then use the `solvate` output (or count the `W` records) to enter the correct water count. The cell below reads the structure and rewrites `[ molecules ]` explicitly.

If you restored a charged checkpoint, it already contains ions; those counts are retained and reconciled in preparation for the next subsection.

In [ ]:
from collections import Counter

def gro_residue_counts(path):
    lines = Path(path).read_text().splitlines()
    atom_count = int(lines[1].strip())
    return Counter(line[5:10].strip() for line in lines[2:2 + atom_count])

def rewrite_molecules(topology, peptide_name, water_count, ion_counts):
    lines = Path(topology).read_text().splitlines()
    # Follow the handout: add the ions include only when ions are present.
    lines = [line for line in lines if "martini_v2.0_ions.itp" not in line]
    if sum(ion_counts.values()):
        main = next(i for i, line in enumerate(lines) if "martini_v2.2.itp" in line)
        lines.insert(main + 1, '#include "martini_v2.0_ions.itp"')

    start = next(i for i, line in enumerate(lines) if line.strip().lower() == "[ molecules ]")
    prefix = lines[:start]
    block = ["[ molecules ]", "; name        number", f"{peptide_name:<16s} 100", f"{'W':<16s} {water_count}"]
    for ion in ("NA+", "CL-"):
        if ion_counts.get(ion, 0):
            block.append(f"{ion:<16s} {ion_counts[ion]}")
    Path(topology).write_text("\n".join(prefix + block).rstrip() + "\n")

system_counts = gro_residue_counts(CG_DIR / f"{PEPTIDE}_water.gro")
ion_counts = Counter({ion: system_counts[ion] for ion in ("NA+", "CL-") if system_counts[ion]})
rewrite_molecules(top_path, PEPTIDE, system_counts["W"], ion_counts)
print("Coordinate counts:", {key: system_counts[key] for key in ("W", "NA+", "CL-")})
print("\nUpdated topology:\n")
print(top_path.read_text())

## 3.7 Interpret the concentration

One hundred peptides in an $8\times8\times8$ nm³ box correspond to about 0.32 M. This is far higher than a typical bulk experiment, but it allows assembly on a short simulation timescale. The handout also notes that solvent evaporation can create much higher local concentrations near a growing nanostructure than the bulk-average concentration.

In [ ]:
import math

n_peptides = 100
volume_nm3 = 8 * 8 * 8
avogadro = 6.02214076e23
volume_litre = volume_nm3 * 1e-24
concentration_molar = (n_peptides / avogadro) / volume_litre
simulated_time_ns = 500_000 * 0.025 / 1000
print(f"Peptide concentration = {concentration_molar:.3f} mol/L")
print(f"Production duration    = {simulated_time_ns:.1f} ns")

> **Question 4 - concentration.** Why does this elevated concentration make aggregation easier to observe? Which conclusions would be unsafe to transfer directly to a dilute experiment?

## 3.8 Add counterions when the peptide is charged

`genion` requires a `.tpr`, so first generate one and then replace water beads with ions:

```bash
gmx grompp -f tripep_water_min.mdp -p TYR-XXX-TYR.top -c TYR-XXX-TYR_water.gro -o TYR-XXX-TYR_genion.tpr
gmx genion -s TYR-XXX-TYR_genion.tpr -pname NA+ -nname CL- -neutral -o TYR-XXX-TYR_water.gro
```

The ion names are explicit because they must match `martini_v2.0_ions.itp`. The output intentionally reuses the water structure name: a high-throughput script can continue with one filename whether the system required ions or not. When prompted by `genion`, select the water group.

In [ ]:
RUN_GENION = RUN_GROMACS_SYSTEM_BUILD  # change only if the box was freshly built above
grompp_ion = [
    "gmx", "grompp", "-f", "tripep_water_min.mdp", "-p", f"{PEPTIDE}.top",
    "-c", f"{PEPTIDE}_water.gro", "-o", f"{PEPTIDE}_genion.tpr",
]
genion = [
    "gmx", "genion", "-s", f"{PEPTIDE}_genion.tpr", "-pname", "NA+",
    "-nname", "CL-", "-neutral", "-o", f"{PEPTIDE}_water.gro",
]
print("Resolved commands:\n", " ".join(grompp_ion), "\n", " ".join(genion))

if RUN_GENION:
    subprocess.run(grompp_ion, cwd=CG_DIR, check=True)
    subprocess.run(genion, cwd=CG_DIR, input="W\n", text=True, check=True)
else:
    print("Ion generation skipped; the 3_Done checkpoint has already been neutralized when necessary.")

## 3.9 Final topology edits after ion replacement

If ions were added, the handout requires two manual edits:

1. add `#include "martini_v2.0_ions.itp"` directly below the main Martini include;
2. update the `W` count and add `NA+` and/or `CL-` lines to `[ molecules ]`.

The ions are collected at the end of the `.gro` file. Re-run the reconciliation cell below after `genion` and inspect both the topology and the final coordinate records.

In [ ]:
system_counts = gro_residue_counts(CG_DIR / f"{PEPTIDE}_water.gro")
ion_counts = Counter({ion: system_counts[ion] for ion in ("NA+", "CL-") if system_counts[ion]})
rewrite_molecules(top_path, PEPTIDE, system_counts["W"], ion_counts)
print(top_path.read_text())
print("Last coordinate records:")
print("\n".join((CG_DIR / f"{PEPTIDE}_water.gro").read_text().splitlines()[-8:]))

## 3.10 Apply the complete setup to all 20 peptides

Only after completing the procedure manually should you inspect and run:

```bash
./setup_tripeptides.sh
```

The script loops over the residue list and automates martinization, insertion, solvation, topology counts, and neutralization.

In [ ]:
setup_script = CG_DIR / "setup_tripeptides.sh"
print(setup_script.read_text())
print("\nNot executed automatically: it requires the legacy martinize/VMD-compatible environment.")

# 4 Running self-assembly simulations

The starting system now contains 100 peptide molecules in Martini water, with counterions where required. Standard minimization and production-MD parameter files are supplied in `4_Running_simulations`.

## 4.1 Inspect and rationalize the MDP files

Pay particular attention to:

- `dt`: integration timestep in ps;
- `nsteps`: number of integration steps;
- `epsilon_r`: relative dielectric constant used by the reaction-field electrostatics;
- `ref_t`: reference temperature for the coupling groups.

The production run uses 500,000 steps of 25 fs, or 12.5 ns. The original handout estimates roughly five minutes on four processors; actual timing depends strongly on hardware and GROMACS version.

In [ ]:
def read_mdp(path):
    values = {}
    for raw in Path(path).read_text().splitlines():
        line = raw.split(";", 1)[0].strip()
        if "=" in line:
            key, value = line.split("=", 1)
            values[key.strip().lower()] = value.strip()
    return values

min_mdp_path = SIM_DIR / "tripep_water_min.mdp"
eq_mdp_path = SIM_DIR / "tripep_water_eq.mdp"
min_mdp = read_mdp(min_mdp_path)
eq_mdp = read_mdp(eq_mdp_path)

for key in ("integrator", "dt", "nsteps", "epsilon_r", "tcoupl", "ref_t", "pcoupl"):
    print(f"{key:12s} minimization={min_mdp.get(key, '-'):18s} production={eq_mdp.get(key, '-')}")
print(f"\nComputed production time: {float(eq_mdp['dt']) * int(eq_mdp['nsteps']) / 1000:.1f} ns")

In [ ]:
# Read the full input files, including the comments that explain the legacy Martini choices.
print("===== tripep_water_min.mdp =====")
print(min_mdp_path.read_text())
print("===== tripep_water_eq.mdp =====")
print(eq_mdp_path.read_text())

> **Question 5 - simulation settings.** Explain `dt`, `nsteps`, `epsilon_r`, and `ref_t` in physical terms. Confirm the total simulated time. Why can a Martini model use a much larger timestep than an unconstrained all-atom peptide model?

## 4.2 Energy minimization

Replace `XXX` and run both commands from `4_Running_simulations`:

```bash
gmx grompp -f tripep_water_min.mdp -p ../3_Coarse-graining/TYR-XXX-TYR.top -c ../3_Coarse-graining/TYR-XXX-TYR_water.gro -o TYR-XXX-TYR_min.tpr -maxwarn 1
gmx mdrun -deffnm TYR-XXX-TYR_min -v
```

## 4.3 Production/equilibration MD

Continue from the minimized structure:

```bash
gmx grompp -f tripep_water_eq.mdp -p ../3_Coarse-graining/TYR-XXX-TYR.top -c TYR-XXX-TYR_min.gro -o TYR-XXX-TYR_eq.tpr -maxwarn 2
gmx mdrun -deffnm TYR-XXX-TYR_eq -v
```

The handout permits the specified warnings for this teaching exercise: the minimization setup may report `CL-` versus `CL`, and the production setup may warn about thermostat/barostat accuracy at the chosen timestep. These warnings would matter when accurate thermodynamics is the scientific objective; do not generalize `-maxwarn` as a routine fix.

In [ ]:
RUN_ONE_SIMULATION = False  # @param {type:"boolean"}

commands = [
    ["gmx", "grompp", "-f", "tripep_water_min.mdp", "-p", f"../3_Coarse-graining/{PEPTIDE}.top", "-c", f"../3_Coarse-graining/{PEPTIDE}_water.gro", "-o", f"{PEPTIDE}_min.tpr", "-maxwarn", "1"],
    ["gmx", "mdrun", "-deffnm", f"{PEPTIDE}_min", "-v"],
    ["gmx", "grompp", "-f", "tripep_water_eq.mdp", "-p", f"../3_Coarse-graining/{PEPTIDE}.top", "-c", f"{PEPTIDE}_min.gro", "-o", f"{PEPTIDE}_eq.tpr", "-maxwarn", "2"],
    ["gmx", "mdrun", "-deffnm", f"{PEPTIDE}_eq", "-v"],
]
print("Resolved commands:")
for command in commands:
    print(" ".join(command))

if RUN_ONE_SIMULATION:
    if shutil.which("gmx") is None:
        raise RuntimeError("GROMACS is not available.")
    for command in commands:
        subprocess.run(command, cwd=SIM_DIR, check=True)
else:
    # Copy only the selected prepared run so the analysis follows the same 4_Running_simulations paths.
    for source in (ROOT / "4_Done").glob(f"{PEPTIDE}_*"):
        if source.is_file():
            shutil.copy2(source, SIM_DIR / source.name)
    print("Restored the selected completed simulation from 4_Done.")

## 4.4 Run the 20-peptide simulation screen

The supplied `run_sims.sh` repeats the four commands for all sequences:

```bash
./run_sims.sh
```

Twenty runs can take too long for a teaching session. The handout therefore recommends `4_Done` when you want to proceed directly to the high-throughput analysis.

In [ ]:
run_script = SIM_DIR / "run_sims.sh"
print(run_script.read_text())
print("\nNot executed automatically. Use 4_Done for the supplied 20-peptide trajectories.")

# 5 Analyzing the results

Replace `XXX` with the selected residue in every command. The order matters: inspect the trajectory, repair periodic-boundary splitting, calculate SASA, isolate the largest cluster, align its principal axes, and finally calculate its moments of inertia.

## 5.1 Inspect the simulation in VMD

The original command is:

```bash
vmd ../4_Running_simulations/TYR-XXX-TYR_min.gro ../4_Running_simulations/TYR-XXX-TYR_eq.xtc
```

A coarse-grained model has no conventional atomistic bonds in this view. In VMD choose **Graphics -> Graphical Representations**, set **Selected Atoms** to `not name W`, and choose **VDW** as the drawing method.

The two Colab views below show the minimized and final snapshots with water/ions hidden. Rotate and zoom both structures.

In [ ]:
print("Minimized dispersed system")
view_gro(SIM_DIR / f"{PEPTIDE}_min.gro", sphere_scale=0.40)
print("Final snapshot (may be split across periodic boundaries)")
view_gro(SIM_DIR / f"{PEPTIDE}_eq.gro", sphere_scale=0.40)

> **Question 6 - visual result.** Do you see general aggregation? Do you see an unambiguous hollow nanotube? Explain your answer using the missing protecting groups, the water/methanol crystallization procedure, and the distinction between a single fibre and a hollow tube.

## 5.2 Center the final snapshot on a large cluster

Aggregates often cross periodic boundaries. The handout sends three selections (`1 1 1`) to `trjconv` and writes a peptide-only centered structure:

```bash
echo 1 1 1 | gmx trjconv -f ../4_Running_simulations/TYR-XXX-TYR_eq.gro -s ../4_Running_simulations/TYR-XXX-TYR_eq.tpr -pbc cluster -center -o TYR-XXX-TYR_clustered.gro
```

In [ ]:
RUN_GROMACS_ANALYSIS = False  # @param {type:"boolean"}

cluster_command = (
    f"echo 1 1 1 | gmx trjconv -f ../4_Running_simulations/{PEPTIDE}_eq.gro "
    f"-s ../4_Running_simulations/{PEPTIDE}_eq.tpr -pbc cluster -center -o {PEPTIDE}_clustered.gro"
)
print("Resolved command:\n", cluster_command)
if RUN_GROMACS_ANALYSIS:
    subprocess.run(cluster_command, cwd=ANALYSIS_DIR, shell=True, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_clustered.gro", ANALYSIS_DIR / f"{PEPTIDE}_clustered.gro")
    print("Restored clustered checkpoint from 5_Done.")

In [ ]:
# Visualization checkpoint: centered peptide cluster, with no solvent in this file.
view_gro(ANALYSIS_DIR / f"{PEPTIDE}_clustered.gro", sphere_scale=0.46)

## 5.3 Quantify aggregation with solvent-accessible surface area

Calculate the peptide SASA after minimization and after assembly using a 0.4 nm probe:

```bash
echo 1 1 | gmx sasa -f ../4_Running_simulations/TYR-XXX-TYR_min.gro -s ../4_Running_simulations/TYR-XXX-TYR_min.tpr -o TYR-XXX-TYR_sasa_init.xvg -surface 'group "Protein"' -probe 0.4
echo 1 1 | gmx sasa -f TYR-XXX-TYR_clustered.gro -s ../4_Running_simulations/TYR-XXX-TYR_eq.tpr -o TYR-XXX-TYR_sasa_end.xvg -surface 'group "Protein"' -probe 0.4
```

Define aggregation propensity as:

$$AP=\frac{SASA_{initial}}{SASA_{final}}$$

A larger ratio means more solvent-accessible surface was buried during assembly.

In [ ]:
sasa_commands = [
    f"echo 1 1 | gmx sasa -f ../4_Running_simulations/{PEPTIDE}_min.gro -s ../4_Running_simulations/{PEPTIDE}_min.tpr -o {PEPTIDE}_sasa_init.xvg -surface 'group \"Protein\"' -probe 0.4",
    f"echo 1 1 | gmx sasa -f {PEPTIDE}_clustered.gro -s ../4_Running_simulations/{PEPTIDE}_eq.tpr -o {PEPTIDE}_sasa_end.xvg -surface 'group \"Protein\"' -probe 0.4",
]
for command in sasa_commands:
    print(command)

if RUN_GROMACS_ANALYSIS:
    for command in sasa_commands:
        subprocess.run(command, cwd=ANALYSIS_DIR, shell=True, check=True)
else:
    for suffix in ("_sasa_init.xvg", "_sasa_end.xvg"):
        shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}{suffix}", ANALYSIS_DIR / f"{PEPTIDE}{suffix}")
    print("Restored SASA checkpoints from 5_Done.")

In [ ]:
def xvg_data(path):
    rows = []
    for line in Path(path).read_text().splitlines():
        if line.strip() and not line.lstrip().startswith(("#", "@")):
            rows.append([float(value) for value in line.split()])
    return rows

sasa_initial = xvg_data(ANALYSIS_DIR / f"{PEPTIDE}_sasa_init.xvg")[-1][1]
sasa_final = xvg_data(ANALYSIS_DIR / f"{PEPTIDE}_sasa_end.xvg")[-1][1]
ap_selected = sasa_initial / sasa_final

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.bar(["Initial", "Final"], [sasa_initial, sasa_final], color=["#59a14f", "#e15759"])
ax.set_ylabel("SASA (nm²)")
ax.set_title(f"{PEPTIDE}: AP = {ap_selected:.3f}")
ax.grid(axis="y", alpha=0.25)
plt.show()
print(f"SASA_initial = {sasa_initial:.3f} nm²")
print(f"SASA_final   = {sasa_final:.3f} nm²")
print(f"AP           = {ap_selected:.3f}")

> **Reference-run note.** If the AP calculated from the selected checkpoint `.xvg` files differs from `sasa_trip.txt` below, report the value you actually calculated and identify its source. The bundled per-peptide checkpoint files and the separate summary table represent supplied reference analyses; stochastic starting configurations and analysis details can change the exact result.

> **Interpretation limit.** SASA burial measures aggregation, not whether the aggregate is soluble, ordered, fibrous, or tubular. A peptide can have high AP because it forms a compact disordered aggregate. Radial distribution functions can probe molecular order; the handout next uses one-dimensionality as a simple shape indicator.

## 5.4 Make a peptide-only index

`options.txt` keeps group 1 and quits. Run:

```bash
gmx make_ndx -f ../4_Running_simulations/TYR-XXX-TYR_eq.gro -o TYR-XXX-TYR_noW.ndx < options.txt
```

In [ ]:
print("options.txt:\n" + (ANALYSIS_DIR / "options.txt").read_text())
make_ndx_command = f"gmx make_ndx -f ../4_Running_simulations/{PEPTIDE}_eq.gro -o {PEPTIDE}_noW.ndx < options.txt"
print("Resolved command:\n", make_ndx_command)
if RUN_GROMACS_ANALYSIS:
    subprocess.run(make_ndx_command, cwd=ANALYSIS_DIR, shell=True, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_noW.ndx", ANALYSIS_DIR / f"{PEPTIDE}_noW.ndx")

## 5.5 Extract the largest cluster with a 0.5 nm cutoff

```bash
gmx clustsize -f ../4_Running_simulations/TYR-XXX-TYR_eq.xtc -s ../4_Running_simulations/TYR-XXX-TYR_eq.tpr -mcn TYR-XXX-TYR_maxclust.ndx -n TYR-XXX-TYR_noW.ndx -cut 0.5
```

The output index contains the bead numbers belonging to the largest cluster in the final trajectory frame.

In [ ]:
clustsize_command = [
    "gmx", "clustsize", "-f", f"../4_Running_simulations/{PEPTIDE}_eq.xtc",
    "-s", f"../4_Running_simulations/{PEPTIDE}_eq.tpr", "-mcn", f"{PEPTIDE}_maxclust.ndx",
    "-n", f"{PEPTIDE}_noW.ndx", "-cut", "0.5",
]
print("Resolved command:\n", " ".join(clustsize_command))
if RUN_GROMACS_ANALYSIS:
    subprocess.run(clustsize_command, cwd=ANALYSIS_DIR, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_maxclust.ndx", ANALYSIS_DIR / f"{PEPTIDE}_maxclust.ndx")

## 5.6 Create a peptide-only run-input file

```bash
gmx convert-tpr -s ../4_Running_simulations/TYR-XXX-TYR_eq.tpr -n TYR-XXX-TYR_noW.ndx -nsteps -1 -o TYR-XXX-TYR_noW.tpr
```

In [ ]:
no_water_tpr_command = [
    "gmx", "convert-tpr", "-s", f"../4_Running_simulations/{PEPTIDE}_eq.tpr",
    "-n", f"{PEPTIDE}_noW.ndx", "-nsteps", "-1", "-o", f"{PEPTIDE}_noW.tpr",
]
print("Resolved command:\n", " ".join(no_water_tpr_command))
if RUN_GROMACS_ANALYSIS:
    subprocess.run(no_water_tpr_command, cwd=ANALYSIS_DIR, input="1\n", text=True, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_noW.tpr", ANALYSIS_DIR / f"{PEPTIDE}_noW.tpr")

## 5.7 Write the largest-cluster coordinates

```bash
gmx trjconv -f TYR-XXX-TYR_clustered.gro -s TYR-XXX-TYR_noW.tpr -n TYR-XXX-TYR_noW.ndx -o TYR-XXX-TYR_maxclust.gro
```

At the interactive prompt, select the peptide group.

In [ ]:
max_cluster_gro_command = [
    "gmx", "trjconv", "-f", f"{PEPTIDE}_clustered.gro", "-s", f"{PEPTIDE}_noW.tpr",
    "-n", f"{PEPTIDE}_noW.ndx", "-o", f"{PEPTIDE}_maxclust.gro",
]
print("Resolved command:\n", " ".join(max_cluster_gro_command))
if RUN_GROMACS_ANALYSIS:
    subprocess.run(max_cluster_gro_command, cwd=ANALYSIS_DIR, input="1\n", text=True, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_maxclust.gro", ANALYSIS_DIR / f"{PEPTIDE}_maxclust.gro")

## 5.8 Create a largest-cluster run-input file

```bash
gmx convert-tpr -s TYR-XXX-TYR_noW.tpr -n TYR-XXX-TYR_maxclust.ndx -nsteps -1 -o TYR-XXX-TYR_maxclust.tpr
```

In [ ]:
max_cluster_tpr_command = [
    "gmx", "convert-tpr", "-s", f"{PEPTIDE}_noW.tpr", "-n", f"{PEPTIDE}_maxclust.ndx",
    "-nsteps", "-1", "-o", f"{PEPTIDE}_maxclust.tpr",
]
print("Resolved command:\n", " ".join(max_cluster_tpr_command))
if RUN_GROMACS_ANALYSIS:
    subprocess.run(max_cluster_tpr_command, cwd=ANALYSIS_DIR, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_maxclust.tpr", ANALYSIS_DIR / f"{PEPTIDE}_maxclust.tpr")

## 5.9 Center and align the cluster with its principal axes

```bash
echo 1 | gmx editconf -f TYR-XXX-TYR_maxclust.gro -princ -c -o TYR-XXX-TYR_princ.gro
```

Comparing aligned structures makes sequence-dependent shape differences easier to see.

In [ ]:
principal_command = f"echo 1 | gmx editconf -f {PEPTIDE}_maxclust.gro -princ -c -o {PEPTIDE}_princ.gro"
print("Resolved command:\n", principal_command)
if RUN_GROMACS_ANALYSIS:
    subprocess.run(principal_command, cwd=ANALYSIS_DIR, shell=True, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_princ.gro", ANALYSIS_DIR / f"{PEPTIDE}_princ.gro")

In [ ]:
# Visualization checkpoint: largest cluster aligned to its principal axes.
view_gro(ANALYSIS_DIR / f"{PEPTIDE}_princ.gro", sphere_scale=0.50)

## 5.10 Calculate the moments of inertia

```bash
echo 1 | gmx gyrate -f TYR-XXX-TYR_princ.gro -s TYR-XXX-TYR_maxclust.tpr -moi -o TYR-XXX-TYR_gyrate.xvg
```

For principal moments ordered as $I_x\le I_y\le I_z$, calculate $I_z/I_x$. A mainly one-dimensional aggregate should satisfy both $I_x\ll I_y$ and $I_y\approx I_z$.

In [ ]:
gyrate_command = f"echo 1 | gmx gyrate -f {PEPTIDE}_princ.gro -s {PEPTIDE}_maxclust.tpr -moi -o {PEPTIDE}_gyrate.xvg"
print("Resolved command:\n", gyrate_command)
if RUN_GROMACS_ANALYSIS:
    subprocess.run(gyrate_command, cwd=ANALYSIS_DIR, shell=True, check=True)
else:
    shutil.copy2(ROOT / "5_Done" / f"{PEPTIDE}_gyrate.xvg", ANALYSIS_DIR / f"{PEPTIDE}_gyrate.xvg")

moi_row = xvg_data(ANALYSIS_DIR / f"{PEPTIDE}_gyrate.xvg")[-1]
time_value, itot, ix, iy, iz = moi_row[:5]
aspect = iz / ix
transverse_mismatch = abs(iy - iz) / ((iy + iz) / 2)
print(f"Ix = {ix:.1f}, Iy = {iy:.1f}, Iz = {iz:.1f}")
print(f"Iz/Ix = {aspect:.3f}")
print(f"relative |Iy-Iz| = {transverse_mismatch:.1%}")

> **Question 7 - shape.** Is $I_z/I_x$ large for the selected peptide, and are $I_y$ and $I_z$ sufficiently similar to support a one-dimensional interpretation? Check the aligned molecular view before calling the object fibrous. Why can neither this ratio nor a snapshot prove that a hollow nanotube formed?

## 5.11 Automate the analysis for all 20 peptides

Once all simulations exist, the handout runs:

```bash
./analysis.sh
```

The script applies the clustering, SASA, largest-cluster, principal-axis, and MOI workflow to every Tyr-X-Tyr sequence. Its summaries are `sasa_trip.txt` and `inertia_trip.txt`.

In [ ]:
analysis_script = ANALYSIS_DIR / "analysis.sh"
print(analysis_script.read_text())
print("\nNot executed automatically. The supplied reference summaries are analyzed below.")

## 5.12 Compare aggregation propensity across the library

The table below is the supplied course summary. Because MD and packing are stochastic, treat the values as one reference screen rather than immutable constants.

In [ ]:
from io import StringIO
import pandas as pd

SASA_SUMMARY = '''pep AP
TYR-ALA-TYR 4.222
TYR-CYS-TYR 4.067
TYR-ASP-TYR 3.670
TYR-GLU-TYR 3.819
TYR-PHE-TYR 3.863
TYR-GLY-TYR 4.150
TYR-HIS-TYR 3.370
TYR-ILE-TYR 5.143
TYR-LYS-TYR 3.767
TYR-LEU-TYR 5.147
TYR-MET-TYR 5.005
TYR-ASN-TYR 4.071
TYR-PRO-TYR 4.133
TYR-GLN-TYR 3.368
TYR-ARG-TYR 3.680
TYR-SER-TYR 3.935
TYR-THR-TYR 3.906
TYR-VAL-TYR 4.061
TYR-TRP-TYR 5.186
TYR-TYR-TYR 3.471
'''
sasa_summary = pd.read_csv(StringIO(SASA_SUMMARY), sep=r"\s+")
sasa_summary["X"] = sasa_summary["pep"].str.split("-").str[1]
sasa_sorted = sasa_summary.sort_values("AP")

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e15759" if x == MIDDLE_RESIDUE else "#4e79a7" for x in sasa_sorted["X"]]
ax.barh(sasa_sorted["X"], sasa_sorted["AP"], color=colors)
ax.set_xlabel(r"Aggregation propensity, $SASA_{initial}/SASA_{final}$")
ax.set_ylabel("Middle residue X")
ax.set_title("Tyr-X-Tyr aggregation screen")
ax.grid(axis="x", alpha=0.25)
plt.show()
display(sasa_sorted.reset_index(drop=True))

> **Question 8 - aggregation ranking.** Which middle residues give the highest and lowest AP? Does the full ranking support a simple hydrophobicity rule, or are there exceptions? Remember that high AP alone can describe disordered insolubility.

## 5.13 Compare one-dimensionality across the library

The supplied moment-of-inertia summary also reports the number of beads (`nr_at`) in the largest cluster. We calculate $I_z/I_x$ and a transverse mismatch $|I_y-I_z|/[(I_y+I_z)/2]$. A convincing 1D candidate combines a large aspect ratio, small transverse mismatch, and a meaningful cluster size.

In [ ]:
INERTIA_SUMMARY = '''pep Time_ps nr_at FrameTime Itot Ix Iy Iz
TYR-ALA-TYR 1.250000e+04 774 0 156791 79675.1 90393.6 100321
TYR-CYS-TYR 1.250000e+04 830 0 247675 66987.6 162767 174248
TYR-ASP-TYR 1.250000e+04 1000 0 887338 98527 611600 635300
TYR-GLU-TYR 1.250000e+04 1000 0 763057 172853 464458 580221
TYR-PHE-TYR 1.250000e+04 1200 0 448217 146829 268718 327308
TYR-GLY-TYR 1.250000e+04 783 0 162956 74010.5 100651 104626
TYR-HIS-TYR 1.250000e+04 1200 0 977382 138066 645227 721037
TYR-ILE-TYR 1.250000e+04 1000 0 271443 119869 158064 185280
TYR-LYS-TYR 1.250000e+04 1100 0 862998 105668 600969 610276
TYR-LEU-TYR 1.250000e+04 1000 0 309848 94152.3 205421 211998
TYR-MET-TYR 1.250000e+04 1000 0 285862 108453 184309 189698
TYR-ASN-TYR 1.250000e+04 1000 0 491689 136567 288191 374237
TYR-PRO-TYR 1.250000e+04 800 0 396832 52159.7 276270 280054
TYR-GLN-TYR 1.250000e+04 510 0 123602 31184.7 80984.2 88014.7
TYR-ARG-TYR 1.250000e+04 1100 0 642226 238533 362801 473215
TYR-SER-TYR 1.250000e+04 1000 0 521459 91047.4 347060 378390
TYR-THR-TYR 1.250000e+04 1000 0 513336 95317.6 345663 367349
TYR-VAL-TYR 1.250000e+04 940 0 277975 89426.2 180295 191746
TYR-TRP-TYR 1.250000e+04 1170 0 672701 145253 421554 503706
TYR-TYR-TYR 1.250000e+04 1200 0 644464 186438 375153 489729
'''
inertia = pd.read_csv(StringIO(INERTIA_SUMMARY), sep=r"\s+")
inertia["X"] = inertia["pep"].str.split("-").str[1]
inertia["Iz_over_Ix"] = inertia["Iz"] / inertia["Ix"]
inertia["Iy_Iz_mismatch"] = (inertia["Iy"] - inertia["Iz"]).abs() / ((inertia["Iy"] + inertia["Iz"]) / 2)
inertia_sorted = inertia.sort_values("Iz_over_Ix")

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#e15759" if x == MIDDLE_RESIDUE else "#76b7b2" for x in inertia_sorted["X"]]
ax.barh(inertia_sorted["X"], inertia_sorted["Iz_over_Ix"], color=colors)
ax.set_xlabel(r"Principal-moment aspect ratio, $I_z/I_x$")
ax.set_ylabel("Middle residue X")
ax.set_title("Largest-cluster one-dimensionality screen")
ax.grid(axis="x", alpha=0.25)
plt.show()
display(inertia.sort_values("Iz_over_Ix", ascending=False)[
    ["pep", "nr_at", "Ix", "Iy", "Iz", "Iz_over_Ix", "Iy_Iz_mismatch"]
].reset_index(drop=True))

> **Question 9 - most fibrous candidates.** Which peptides combine a high $I_z/I_x$ with $I_y\approx I_z$? Check `nr_at` and the aligned structures before deciding. How might a high aspect ratio from a small cluster mislead you?

## 5.14 Combine aggregation and shape

Aggregation propensity and one-dimensionality answer different questions. The scatter plot makes that distinction explicit; point size reflects the largest-cluster bead count and color reflects the agreement between the two transverse moments.

In [ ]:
combined = sasa_summary.merge(
    inertia[["pep", "nr_at", "Iz_over_Ix", "Iy_Iz_mismatch"]], on="pep", how="inner"
)
combined["X"] = combined["pep"].str.split("-").str[1]

fig, ax = plt.subplots(figsize=(9, 6.5))
scatter = ax.scatter(
    combined["AP"], combined["Iz_over_Ix"],
    s=combined["nr_at"] * 0.16,
    c=combined["Iy_Iz_mismatch"], cmap="viridis_r",
    edgecolor="black", linewidth=0.5, alpha=0.85,
)
for _, row in combined.iterrows():
    ax.annotate(row["X"], (row["AP"], row["Iz_over_Ix"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Aggregation propensity, AP")
ax.set_ylabel(r"One-dimensionality, $I_z/I_x$")
ax.set_title("Aggregation strength is not aggregate shape")
ax.grid(alpha=0.25)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label(r"Relative $I_y$-$I_z$ mismatch")
plt.show()

> **Question 10 - integrated interpretation.** Is the strongest aggregator also the most one-dimensional? Identify at least two sequences that demonstrate why AP and MOI must be interpreted together. Use the molecular views to decide whether the numerical result is structurally plausible.

# Discussion and submission checklist

Your report should include:

- the selected Tyr-X-Tyr sequence and a justified prediction for X;
- confirmation that 20 atomistic structures were created or verified;
- the selected peptide's Martini `[ atoms ]` mapping;
- the exact peptide, water, and ion counts in the final topology;
- an explanation of the ~0.32 M concentration and 12.5 ns duration;
- initial, final, clustered, and principal-axis-aligned structural views;
- $SASA_{initial}$, $SASA_{final}$, and AP for the selected peptide;
- $I_x$, $I_y$, $I_z$, $I_z/I_x$, transverse-moment agreement, and largest-cluster size;
- the most strongly aggregating and most fibrous candidates in the 20-peptide screen;
- limitations: coarse graining, high concentration, short sampling, one trajectory per sequence, omitted protecting groups, aqueous conditions, and the difference between aggregation, fibres, and hollow nanotubes.

# References and source material

- Ray, S., Haldar, D., Drew, M. G. B. & Banerjee, A. *A New Motif in the Formation of Peptide Nanotubes: The Crystallographic Signature.* **Organic Letters** 6, 4463-4465 (2004).
- Ray, S., Drew, M. G. B., Das, A. K. & Banerjee, A. *The role of terminal tyrosine residues in the formation of tripeptide nanotubes: a crystallographic insight.* **Tetrahedron** 62, 7274-7283 (2006).
- Course handout: *Molecular physical pharmacy, 3FC003 - Molecular dynamics exercise, 2020: High throughput peptide self-assembly*.
- The handout acknowledges the Martini tutorial at `cgmartini.nl` as the basis of the exercise.